# Standing waves on a string

*waves · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/standing_waves)

The 1D wave equation on a string with both ends fixed. The second derivative becomes a matrix (the finite-difference stencil $D_{ij}$), so one einsum moves every point of several strings at once: the first harmonics and a plucked string.

**Vector form**

$$\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2},\qquad u(0,t) = u(L,t) = 0$$

**Index (tensor) form**

$$\ddot u_{bi} = c^2\,D_{ij}\,u_{bj},\qquad D_{ij} = \frac{\delta_{i+1,j} - 2\,\delta_{ij} + \delta_{i-1,j}}{\Delta x^2},\qquad f_n = \frac{n\,c}{2L}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_{bi}$ | `u` | (B, M) | displacement of string $b$ at interior grid point $i$ (the two fixed ends are left out) |
| $D_{ij}$ | `stencil(M, dx, dev)` | (M, M) | second-derivative matrix: $-2$ on the diagonal, $1$ next to it |
| $D_{ij}u_{bj}$ | `einsum('ij,bj->bi', D, u)` | (B, M) | repeated $j$ ⇒ summed: $\partial_x^2 u$ for every string |
| $u^{n+1} = 2u^n - u^{n-1} + \Delta t^2\,\ddot u^n$ | `leapfrog_step(u, u_prev, D, c, dt)` | (B, M) | centred difference in time (leap-frog) |
| $f_n$ | `mode_frequencies(D, c)` | (M,) | eigenvalues of $D$ give $\omega_n^2 = -c^2\lambda_n$: the harmonics |

Rows 1 to $n$ start in the shape of a single harmonic $\sin(n\pi x/L)$ and just breathe in place at $f_n = nc/2L$: a standing wave. The plucked string (a triangle) is a sum of many harmonics, so its kink splits into two pieces that run to the ends, flip over and come back. After one fundamental period $2L/c$ every shape is back where it started. The time step obeys the Courant condition $c\,\Delta t \le \Delta x$; the energy $\tfrac12\int (\dot u^2 + c^2 u_x^2)\,dx$ (per unit mass density) stays constant because nothing damps the string.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/standing_waves/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    L, c = params["L"], params["c"]
    x, dx = tensors.grid(L, params["n_points"], dev)
    D = tensors.stencil(len(x) - 2, dx, dev)
    u0 = tensors.initial_shapes(x, L, params["n_harmonics"], params["pluck_at"])

    t_max = params["periods"] * 2 * L / c  # fundamental period 2L/c
    times, traj, dt = solve.solve(u0, D, c, dx, t_max)

    freqs = equation.mode_frequencies(D, c)[: params["n_harmonics"]].tolist()
    for n, f in enumerate(freqs, start=1):
        print(f"harmonic {n}:  f = {f:.4f} Hz   (theory n c / 2L = {n * c / (2 * L):.4f} Hz)")
    E = equation.energy(traj, dx, dt, c)  # (T-2, B)
    drift = ((E - E[0]).abs().max(0).values / E[0]).max().item()
    print(f"largest relative energy change: {drift:.2e}")
    return plot.animate(times, x, traj, freqs, out_dir / "standing_waves.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/standing_waves/equation.py
r"""Physics → tensor translation.

The wave equation for a string with fixed ends,

    \partial_t^2 u = c^2 \partial_x^2 u,       u(0, t) = u(L, t) = 0,

is discretised on M interior grid points. The second derivative becomes a matrix D_{ij} (the stencil
1, −2, 1 divided by Δx²), and B strings are moved together:

    \ddot u_{bi} = c^2 D_{ij} u_{bj}        (j is repeated ⇒ summed)

Time is discretised with the centred difference (leap-frog):

    u^{n+1}_{bi} = 2 u^n_{bi} - u^{n-1}_{bi} + \Delta t^2 \ddot u^n_{bi}

The eigenvalues λ_n of D give the normal modes: ω_n² = −c² λ_n, close to (nπc/L)².
"""

import math

import torch


def acceleration(u: torch.Tensor, D: torch.Tensor, c: float) -> torch.Tensor:
    r"""\ddot u_{bi} = c^2 D_{ij} u_{bj}   — j summed.   u: (B, M), D: (M, M) → (B, M)"""
    return c**2 * torch.einsum("ij,bj->bi", D, u)


def leapfrog_step(u: torch.Tensor, u_prev: torch.Tensor, D: torch.Tensor, c: float, dt: float) -> torch.Tensor:
    """u^{n+1} = 2 u^n − u^{n−1} + Δt² ü^n.   All tensors (B, M)."""
    return 2 * u - u_prev + dt**2 * acceleration(u, D, c)


def mode_frequencies(D: torch.Tensor, c: float) -> torch.Tensor:
    """Frequencies f_n = ω_n / 2π of the discrete string, lowest first: (M,)."""
    lam = torch.linalg.eigvalsh(D)  # all negative
    omega = c * torch.sqrt(-lam)
    return torch.sort(omega).values / (2 * math.pi)


def energy(traj: torch.Tensor, dx: float, dt: float, c: float) -> torch.Tensor:
    r"""Energy per unit mass density  E_b = ½ Σ_i (\dot u_{bi}² + c² (∂_x u)_{bi}²) Δx.

    traj: (T, B, N) including the fixed ends. The velocity uses centred differences in time,
    so the result has T − 2 rows: (T − 2, B).
    """
    v = (traj[2:] - traj[:-2]) / (2 * dt)  # (T-2, B, N)
    ux = (traj[1:-1, :, 1:] - traj[1:-1, :, :-1]) / dx  # (T-2, B, N-1) slope between grid points
    kinetic = 0.5 * torch.einsum("tbi,tbi->tb", v, v) * dx
    potential = 0.5 * c**2 * torch.einsum("tbi,tbi->tb", ux, ux) * dx
    return kinetic + potential


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/standing_waves/tensors.py
"""Build the concrete tensors: the grid, the stencil matrix and the starting shapes of the strings."""

import math

import torch


def grid(L: float, n_points: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """x of shape (N,) from 0 to L, ends included, and the spacing Δx."""
    x = torch.linspace(0.0, L, n_points, device=device)
    return x, L / (n_points - 1)


def stencil(m: int, dx: float, device: torch.device) -> torch.Tensor:
    """D_{ij} of shape (M, M): (δ_{i+1,j} − 2δ_{ij} + δ_{i−1,j}) / Δx².

    The fixed ends u = 0 are built in: the first and last rows simply miss one neighbour.
    """
    main = -2.0 * torch.ones(m, device=device)
    side = torch.ones(m - 1, device=device)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / dx**2


def initial_shapes(x: torch.Tensor, L: float, n_harmonics: int, pluck_at: float) -> torch.Tensor:
    """u0 of shape (B, N) with B = n_harmonics + 1.

    rows 0 … H−1:  sin(nπx/L) for n = 1 … H     (single harmonics)
    row  H:        a triangle with its peak at x_p = pluck_at · L    (plucked string)
    """
    n = torch.arange(1, n_harmonics + 1, device=x.device, dtype=x.dtype)  # (H,)
    harmonics = torch.sin(math.pi * n[:, None] * x[None, :] / L)  # (H, N)
    xp = pluck_at * L
    pluck = torch.where(x < xp, x / xp, (L - x) / (L - xp))  # (N,) height 1 at x_p, 0 at both ends
    return torch.cat([harmonics, pluck[None, :]], dim=0)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/standing_waves/solve.py
"""March the strings forward with leap-frog; the time step obeys the Courant condition c Δt ≤ Δx."""

import torch

from problems._common.progress import steps

from .equation import acceleration, leapfrog_step


def solve(
    u0: torch.Tensor, D: torch.Tensor, c: float, dx: float, t_max: float, courant: float = 0.9
) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Return (times (T,), trajectory (T, B, N) with the fixed ends, dt).

    u0: (B, N) starting shapes, released from rest.
    """
    dt = courant * dx / c
    n_steps = int(round(t_max / dt))
    u = u0[:, 1:-1]  # (B, M) interior points only
    # Released from rest: choose u^{-1} so that the first step is u^1 = u^0 + ½ Δt² ü^0
    u_prev = u + 0.5 * dt**2 * acceleration(u, D, c)

    frames = [u]
    for _ in steps(n_steps, "time steps"):
        u, u_prev = leapfrog_step(u, u_prev, D, c, dt), u
        frames.append(u)

    interior = torch.stack(frames)  # (T, B, M)
    traj = torch.nn.functional.pad(interior, (1, 1))  # put the fixed ends u = 0 back: (T, B, N)
    times = torch.arange(n_steps + 1, device=u0.device) * dt
    return times, traj, dt


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/standing_waves/plot.py
"""Animate every string in its own row: harmonics breathe in place, the plucked string's kink runs back and forth."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(times: torch.Tensor, x: torch.Tensor, traj: torch.Tensor, freqs: list[float], path: Path) -> Path:
    t = times.cpu().numpy()
    xs = x.cpu().numpy()
    u = traj.cpu().numpy()  # (T, B, N)
    n_rows = u.shape[1]

    fig, axes = plt.subplots(n_rows, 1, figsize=(7, 1.1 + 1.0 * n_rows), sharex=True, squeeze=False)
    lines = []
    for b, ax in enumerate(axes[:, 0]):
        color = ACCENT[b % len(ACCENT)]
        ax.plot(xs, u[0, b], ls="--", color="#9ca3af", lw=1)  # starting shape
        if b < n_rows - 1:  # a harmonic swings between +u0 and −u0
            ax.plot(xs, -u[0, b], ls="--", color="#9ca3af", lw=1)
        lines.append(ax.plot(xs, u[0, b], color=color, lw=2)[0])
        ax.plot([xs[0], xs[-1]], [0, 0], "o", color="#374151", ms=4)  # fixed ends
        ax.set_ylim(-1.25, 1.25)
        ax.set_yticks([-1, 0, 1])
        name = "plucked" if b == n_rows - 1 else f"n = {b + 1},  f = {freqs[b]:.3f} Hz"
        ax.text(0.01, 0.92, name, transform=ax.transAxes, va="top", fontsize=9)
    axes[-1, 0].set_xlabel("x (m)")
    title = axes[0, 0].set_title("string displacement u(x, t)")  # placeholder so tight_layout leaves room

    def update(step: int):
        for b, line in enumerate(lines):
            line.set_ydata(u[step, b])
        title.set_text(f"string displacement u(x, t)    t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_harmonics': 3,
    'pluck_at': 0.25,
    'c': 1.0,
    'L': 1.0,
    'n_points': 101,
    'periods': 2.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.standing_waves")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.standing_waves.main")
problem_dir = Path("problems/waves/standing_waves")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
